In [0]:
%pip install pyyaml 

In [0]:
dbutils.library.restartPython()

In [0]:
import yaml
import os
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, to_date


In [0]:
orders_df = spark.read.parquet("/Volumes/pei/bronze/orders/")
customers_df = spark.read.parquet("/Volumes/pei/bronze/customer/")
products_df = spark.read.parquet("/Volumes/pei/bronze/product/")

In [0]:
def read_source(source_config):
    return (
        spark.read
        .format(source_config["format"])
        .load(source_config["path"])
    )

In [0]:
def apply_schema_and_rename(df, schema_config):
    for col_def in schema_config:
        source_col = col_def["source_name"]
        target_col = col_def["name"]
        dtype = col_def["type"]

        if source_col in df.columns:
            df = df.withColumnRenamed(source_col, target_col)
            df = df.withColumn(target_col, F.col(target_col).cast(dtype))
        else:
            raise Exception(f"Column {source_col} not found in source")

    return df

In [0]:
# Function to transform columns datatypes
def apply_transformations(df, transformations):

    for t in transformations:
        t_type = t["type"]

        if t_type == "cast":
            for col_name, dtype in t["columns"].items():

                # If dtype is simple (string)
                if isinstance(dtype, str):
                    df = df.withColumn(col_name, col(col_name).cast(dtype))

                # If dtype is dict (has format)
                elif isinstance(dtype, dict):
                    if dtype["type"] == "date":
                        df = df.withColumn(
                            col_name,
                            to_date(col(col_name), dtype["format"])
                        )

    return df

In [0]:
#Function to take input dataframe, config and apply deduplication logic
def apply_deduplication(df, dedup_config):
    if not dedup_config:
        return df

    keys = dedup_config["keys"]
    order_col = dedup_config["order_by"]

    window = Window.partitionBy(*keys).orderBy(F.col(order_col).desc())

    df = (
        df.withColumn("rn", F.row_number().over(window))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

    return df

In [0]:
# Function to take input as dataframe and the list of columns it will retain and returns the transformed dataframe
def select_columns(df, columns):
    return df.select(*columns)

In [0]:
# Function to take dataframe, target config in input and writes the dataframe to target location
def write_target(df, target_config):
    writer = (
        df.write
        .format(target_config["format"])
        .mode(target_config["mode"])
    )

    # Optional partitioning, in case of future scaling. Ex: partition orders data by year
    if "partition_by" in target_config:
        writer = writer.partitionBy(*target_config["partition_by"])

    # Write to path
    if "path" in target_config:
        writer.save(target_config["path"])

In [0]:
def apply_quality_checks(df, rules):
    for rule in rules:
        col = rule["column"]
        condition = rule["rule"]

        if condition == "not_null":
            df = df.filter(F.col(col).isNotNull())
        else:
            df = df.filter(f"{col} {condition}")

    return df

In [0]:
def process_table(config):

    # Read data from the source
    df = read_source(config["source"])
    
    #Impose schema and rename columns
    df = apply_schema_and_rename(df, config["schema"])
    
    # Apply transformations
    df = apply_transformations(df, config.get("transformations", []))

    #Deduplicate data 
    df = apply_deduplication(df, config.get("deduplication"))

    #Apply data Quality rules
    df = apply_quality_checks(df, config.get("quality_checks", []))
    
    #Bring only required columns
    df = select_columns(df, config["columns"]["select"])
    
    #Load data in Silver layer
    write_target(df, config["target"])
    

In [0]:
# CONFIG_DIR : all configs files in silver folder
CONFIG_DIR = "/Workspace/Users/harkiratkr8@gmail.com/PEI_ProfitAnalysis/configs/silver/"

#Parse through all config files of .yaml nomenclature
def load_all_configs(config_dir):
    configs = []

    for file in os.listdir(config_dir):
        if file.endswith(".yaml"):
            full_path = os.path.join(config_dir, file)
            with open(full_path, "r") as f:
                cfg = yaml.safe_load(f)

            cfg["_config_path"] = full_path
            configs.append(cfg)

    return configs

In [0]:
#Parsing through all config yaml files and processing
configs = load_all_configs(CONFIG_DIR)

for cfg in configs:
    try:
        process_table(cfg)

    except Exception as e:
        print(f"Failed: {cfg.get('target', {}).get('table')} → {str(e)}")